# Étape 1 : Créer la grille de mailles

Table **Maille** du diagramme (champs de cette étape) :

| Champ | Type | Remarque |
|---|---|---|
| `id` | int {PK} | |
| `geom` | Polygon [26191] | |
| `centroide` | Point [26191] | un seul champ (pas de x / y) |
| `taille_m` | int | 100 (ou 50 pour le test) |
| `commune_id` | int {FK} | = `Commune.id` |

Les autres champs de `Maille` (pente_moy, hand_moy, twi_moy, flux_max, depression_moy, pct_impermeable, ruissel_moy, densite_bati, densite_routes, distance_oued, indice_equip, distance_secours) sont remplis dans les étapes suivantes.

1. Générer une grille de 100 m sur l'emprise de la zone d'étude
2. Garder les mailles qui intersectent les communes de `limite_casablanca_settat`
3. Calculer le centroïde de chaque maille
4. Attribuer `commune_id` : la commune qui contient le centroïde (une seule commune par maille)

## 1. Imports et chemins

In [4]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import shapely

# --- Racine du projet (le notebook peut être lancé depuis /scripts)
ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent

# --- Données d'entrée : limites des communes (reprojetées en mètres)
# ⚠️ Adaptez le nom du fichier s'il est différent chez vous
DOSSIER_CLEAN = ROOT / "donnees_traitees" / "casablanca_settat_clean"
FICHIER_COMMUNES = DOSSIER_CLEAN / "communes_casablanca_settat_clean.gpkg"

# --- Sortie : la grille de mailles
DOSSIER_SORTIE = ROOT / "donnees_traitees" / "mailles"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

# --- Paramètres
TAILLE_MAILLE = 100                 # taille de la maille en mètres (50 pour le test à 50 m)
COL_COMMUNE = "commune_id"          # ⚠️ nom de la colonne identifiant la commune
FICHIER_MAILLES = DOSSIER_SORTIE / f"mailles_{TAILLE_MAILLE}m.gpkg"
# Système de coordonnées du projet
CRS_PROJET = 26191
print("ROOT            :", ROOT)
print("FICHIER_COMMUNES:", FICHIER_COMMUNES, "->", FICHIER_COMMUNES.exists())
print("FICHIER_MAILLES :", FICHIER_MAILLES)

ROOT            : c:\Users\dell\Documents\geoai-inondation-casablanca
FICHIER_COMMUNES: c:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\casablanca_settat_clean\communes_casablanca_settat_clean.gpkg -> True
FICHIER_MAILLES : c:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\mailles_100m.gpkg


In [2]:
pip install scikit-image

   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.8 MB 2.8 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/12.8 MB 2.8 MB/s eta 0:00:05
   ----- ---------------------------------- 1.8/12.8 MB 3.0 MB/s eta 0:00:04
   ------- -------------------------------- 2.4/12.8 MB 3.0 MB/s eta 0:00:04
   --------- ------------------------------ 2.9/12.8 MB 2.9 MB/s eta 0:00:04
   ----------- ---------------------------- 3.7/12.8 MB 3.0 MB/s eta 0:00:04
   ------------- -------------------------- 4.5/12.8 MB 3.0 MB/s eta 0:00:03
   --------------- ------------------------ 5.0/12.8 MB 3.0 MB/s eta 0:00:03
   ------------------ --------------------- 5.8/12.8 MB 3.0 MB/s eta 0:00:03
   ------------------- -------------------- 6.3/12.8 MB 3.0 MB/s eta 0:00:03
   ---------------------- ----------------- 7.1/12.8 MB 3.0 MB/s eta 0:00:02
   ----------------------- ---------------- 7.6/12.8 MB 3.0 MB/s eta 0:00:02
   ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 2. Lecture des communes

Dans le diagramme, la table **Commune** a les champs `id`, `code_commune`, `nom`, `geom`, `densite_hab_km2`.
`Maille.commune_id` est la clé étrangère vers `Commune.id`.

In [17]:
communes = gpd.read_file(FICHIER_COMMUNES)

# Le diagramme impose EPSG:26191 (en mètres)
assert communes.crs is not None, "La couche des communes n'a pas de CRS."
if communes.crs.to_epsg() != CRS_PROJET:
    communes = communes.to_crs(CRS_PROJET)

# Identifiant de la commune = Commune.id (PK du diagramme)
# Si la couche n'a pas de colonne "id", on en crée une (1..n) : à reprendre à l'identique
# quand la table Commune sera créée dans la base.
if "id" not in communes.columns:
    communes["id"] = np.arange(1, len(communes) + 1)
    print("⚠️ Colonne 'id' absente : créée de 1 à n.")
assert communes["id"].is_unique, "Commune.id doit être unique"

print("CRS :", communes.crs)
print("Nombre de communes :", len(communes))
print("Colonnes :", list(communes.columns))
communes[["id", "geometry"]].head()

CRS : EPSG:26191
Nombre de communes : 153
Colonnes : ['id', 'osm_id', 'nom', 'surface_km2', 'geometry']


,id,geometry
0,1,"MULTIPOLYGON (((329629.602 300777.479, 329784...."
1,2,"MULTIPOLYGON (((223754.51 302130.167, 224035.8..."
2,3,"MULTIPOLYGON (((281901.197 222354.017, 281924...."
3,4,"MULTIPOLYGON (((322357.316 271330.667, 322380...."
4,5,"MULTIPOLYGON (((304947.221 340050.746, 304988...."


## 3. Génération de la grille sur l'emprise

In [18]:
# Emprise de la zone d'étude (xmin, ymin, xmax, ymax)
xmin, ymin, xmax, ymax = communes.total_bounds

# On aligne l'origine sur un multiple de la taille de maille pour avoir une grille « propre »
xmin = np.floor(xmin / TAILLE_MAILLE) * TAILLE_MAILLE
ymin = np.floor(ymin / TAILLE_MAILLE) * TAILLE_MAILLE
xmax = np.ceil(xmax / TAILLE_MAILLE) * TAILLE_MAILLE
ymax = np.ceil(ymax / TAILLE_MAILLE) * TAILLE_MAILLE

# Coordonnées du coin inférieur gauche de chaque maille
xs = np.arange(xmin, xmax, TAILLE_MAILLE)
ys = np.arange(ymin, ymax, TAILLE_MAILLE)
xx, yy = np.meshgrid(xs, ys)
xx, yy = xx.ravel(), yy.ravel()

# Création vectorisée des carrés (rapide, même pour des millions de mailles)
carres = shapely.box(xx, yy, xx + TAILLE_MAILLE, yy + TAILLE_MAILLE)

grille = gpd.GeoDataFrame(geometry=carres, crs=communes.crs)
print(f"Grille brute : {len(grille):,} mailles")

Grille brute : 3,632,382 mailles


## 4. Conserver les mailles qui intersectent les communes (`id`, `geom`, `taille_m`)

In [19]:
# Jointure spatiale "intersects" : on garde les mailles qui touchent au moins une commune
# (une maille à cheval sur deux communes apparaît deux fois, on dédoublonne ensuite)
idx_garde = gpd.sjoin(
    grille, communes[["geometry"]], how="inner", predicate="intersects"
).index.unique()

mailles = grille.loc[idx_garde].copy().reset_index(drop=True)

# Noms du diagramme : id, geom, taille_m
mailles = mailles.rename_geometry("geom")
mailles["id"] = np.arange(1, len(mailles) + 1)
mailles["taille_m"] = TAILLE_MAILLE

print(f"Mailles conservées : {len(mailles):,}")

Mailles conservées : 2,026,506


## 5. Centroïde de chaque maille (un seul champ `centroide`, type Point)

In [20]:
# Un seul champ "centroide" (géométrie Point, EPSG:26191), pas de centroide_x / centroide_y
mailles["centroide"] = mailles["geom"].centroid

mailles[["id", "taille_m", "centroide"]].head()

,id,taille_m,centroide
0,1,100,POINT (336550 192950)
1,2,100,POINT (336650 192950)
2,3,100,POINT (336750 192950)
3,4,100,POINT (336350 193050)
4,5,100,POINT (336450 193050)


## 6. Attribution de `commune_id` (= `Commune.id`) par le centroïde

In [21]:
# Relance possible : on repart sans colonne commune_id
if "commune_id" in mailles.columns:
    mailles = mailles.drop(columns="commune_id")

# Points = centroïdes, avec l'id de la maille
pts = gpd.GeoDataFrame(
    mailles[["id"]],
    geometry=mailles["centroide"].values,
    crs=mailles.crs,
)

# Communes : on renomme Commune.id en commune_id pour éviter le conflit avec Maille.id
communes_ref = communes[["id", "geometry"]].rename(columns={"id": "commune_id"})

# Commune qui CONTIENT le centroïde
pts_commune = gpd.sjoin(pts, communes_ref, how="left", predicate="within")

# Centroïde pile sur une limite : on garde une seule commune
pts_commune = pts_commune.drop_duplicates(subset="id", keep="first")

# Rattachement à la table des mailles
mailles = mailles.merge(pts_commune[["id", "commune_id"]], on="id", how="left")
mailles = gpd.GeoDataFrame(mailles, geometry="geom", crs=pts.crs)

# --- Diagnostic AVANT suppression
sans_commune = mailles[mailles["commune_id"].isna()]
print(f"Mailles sans commune (centroïde hors limite) : {len(sans_commune):,}")

if len(sans_commune) > 0:
    zone = communes.geometry.union_all()   # (ou .unary_union avec une ancienne version de geopandas)
    dist = gpd.GeoSeries(sans_commune["centroide"], crs=mailles.crs).distance(zone)
    print(dist.describe())
    print(f"Centroïdes à plus de 100 m de la zone : {(dist > 100).sum():,}")

# --- Suppression des mailles sans commune (règle : une seule commune par maille)
mailles = mailles.dropna(subset=["commune_id"]).reset_index(drop=True)
mailles["commune_id"] = mailles["commune_id"].astype(int)

print(f"Mailles finales : {len(mailles):,}")

Mailles sans commune (centroïde hors limite) : 5,801
count    5801.000000
mean       32.065176
std        18.941018
min         0.010317
25%        15.757488
50%        31.696848
75%        47.777669
max        70.563512
dtype: float64
Centroïdes à plus de 100 m de la zone : 0
Mailles finales : 2,020,705


## 7. Contrôles et sauvegarde

Un GeoPackage n'accepte qu'**une seule géométrie active par couche**. Pour garder un seul champ `centroide` dans la même table `maille` que `geom`, le centroïde est enregistré en **WKT** (texte). Pour le relire en géométrie :
`gpd.GeoSeries.from_wkt(maille["centroide"], crs=26191)`.

In [22]:
# Champs et ordre exacts du diagramme (étape 1) : id, geom, centroide, taille_m, commune_id
mailles = mailles[["id", "geom", "centroide", "taille_m", "commune_id"]]
mailles["id"] = mailles["id"].astype(int)
mailles["taille_m"] = mailles["taille_m"].astype(int)
mailles["commune_id"] = mailles["commune_id"].astype(int)

# --- Contrôles
assert list(mailles.columns) == ["id", "geom", "centroide", "taille_m", "commune_id"]
assert mailles["id"].is_unique, "id doit être unique"
assert mailles["commune_id"].isin(communes["id"]).all(), "commune_id doit exister dans Commune.id"
assert (mailles["taille_m"] == TAILLE_MAILLE).all()
assert (mailles["geom"].geom_type == "Polygon").all()
assert (mailles["centroide"].geom_type == "Point").all()
assert (mailles["geom"].area == TAILLE_MAILLE ** 2).all(), "Surface incorrecte"
assert mailles.crs.to_epsg() == CRS_PROJET, "CRS doit être EPSG:26191"

# --- Sauvegarde : une seule couche "maille" ; centroide enregistré en WKT (texte)
export = mailles.copy()
export["centroide"] = shapely.to_wkt(export["centroide"].values, rounding_precision=3)
export.to_file(FICHIER_MAILLES, layer="maille", driver="GPKG")

print("Fichier écrit :", FICHIER_MAILLES)
print("Champs :", list(export.columns))
export.head()

Fichier écrit : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\mailles_100m.gpkg
Champs : ['id', 'geom', 'centroide', 'taille_m', 'commune_id']


,id,geom,centroide,taille_m,commune_id
0,2,"POLYGON ((336700 192900, 336700 193000, 336600...",POINT (336650 192950),100,62
1,6,"POLYGON ((336600 193000, 336600 193100, 336500...",POINT (336550 193050),100,62
2,7,"POLYGON ((336700 193000, 336700 193100, 336600...",POINT (336650 193050),100,62
3,8,"POLYGON ((336800 193000, 336800 193100, 336700...",POINT (336750 193050),100,62
4,10,"POLYGON ((336400 193100, 336400 193200, 336300...",POINT (336350 193150),100,62


## Étape 2 : densite_bati (overlay par intersection)
Sortie : Fichier CSV stocké dans donnees_traitees/mailles/densite_bati_100m.csv

In [28]:
# --- 1. Chemins et Configuration

FICHIER_BATI = DOSSIER_CLEAN / "bati_casablanca_settat_clean.gpkg"
TAILLE_MAILLE = 100                      # 50 pour le test à 50 m
DOSSIER_MAILLES = ROOT / "donnees_traitees" / "mailles"
FICHIER_MAILLES = DOSSIER_MAILLES / f"mailles_{TAILLE_MAILLE}m.gpkg"
FICHIER_CSV_SORTIE = DOSSIER_MAILLES / "densite_bati_100m.csv"

print("Lecture des données...")

# Lecture des mailles
mailles = gpd.read_file(FICHIER_MAILLES, layer="maille")

# Correction du nom de la colonne géométrique
if "geometry" in mailles.columns and "geom" not in mailles.columns:
    mailles = mailles.rename_geometry("geom")

elif "geom" in mailles.columns and mailles.geometry.name != "geom":
    mailles = mailles.set_geometry("geom")

# Lecture du bâti
bati = gpd.read_file(FICHIER_BATI)

if bati.crs.to_epsg() != CRS_PROJET:
    bati = bati.to_crs(CRS_PROJET)

print(f"Nombre de mailles : {len(mailles):,}")
print(f"Nombre de bâtiments : {len(bati):,}")

Lecture des données...
Nombre de mailles : 2,020,705
Nombre de bâtiments : 84,962


In [29]:
# --- 2. Filtre spatial préliminaire pour optimiser l'intersection
bbox_mailles = mailles.total_bounds
bati_filtre = bati.cx[
    bbox_mailles[0] : bbox_mailles[2], bbox_mailles[1] : bbox_mailles[3]
].copy()

mailles_geom = mailles[["id", "geom"]].copy()
bati_geom = bati_filtre[["geometry"]].copy()

# --- 3. Overlay par Intersection
print("Calcul de l'intersection (overlay)...")
# Chaque morceau découpé prend l'attribut 'id' de la maille qu'il recouvre
morceaux_bati = gpd.overlay(
    bati_geom, mailles_geom, how="intersection", keep_geom_type=True
)

# --- 4. Calcul de la surface réelle de chaque morceau découpé
# ⚠️ Consigne : calcul avec .area et non le surface_m2 d'origine
morceaux_bati["surf_morceau_m2"] = morceaux_bati.geometry.area

# --- 5. Somme des surfaces des morceaux par maille
print("Agrégation des surfaces par maille...")
surf_par_maille = morceaux_bati.groupby("id")["surf_morceau_m2"].sum()

# --- 6. Calcul de la densité par maille (entre 0 et 1)
surface_maille = mailles["geom"].area
surface_bati_totale = mailles["id"].map(surf_par_maille).fillna(0.0)
# Ratio : surface bâtie divisée par surface de la maille
densite_bati = (surface_bati_totale / surface_maille).clip(
    lower=0.0, upper=1.0
)

Calcul de l'intersection (overlay)...
Agrégation des surfaces par maille...


In [30]:
import pandas as pd
# --- 7. Création du DataFrame de sortie (id, densite_bati)
df_sortie = pd.DataFrame({
    "id": mailles["id"].astype(int),
    "densite_bati": densite_bati.round(6),  # 6 décimales pour la précision
})

# --- 8. Contrôles et assertions métier
print("\n--- Contrôles de validation ---")
print("Nombre total de lignes :", len(df_sortie))
print("Valeurs NaN :", df_sortie["densite_bati"].isna().sum())
print(
    f"Min : {df_sortie['densite_bati'].min():.4f} | Max :"
    f" {df_sortie['densite_bati'].max():.4f}"
)
print(f"Moyenne : {df_sortie['densite_bati'].mean():.4f}")
print(
    "Mailles avec bâti :"
    f" {(df_sortie['densite_bati'] > 0).sum():,} / {len(df_sortie):,} ("
    f" {(df_sortie['densite_bati'] > 0).mean() * 100:.1f}%)"
)

assert (
    df_sortie["densite_bati"].isna().sum() == 0
), "Il ne doit pas y avoir de NaN"
assert (
    (df_sortie["densite_bati"] >= 0.0) & (df_sortie["densite_bati"] <= 1.0)
).all(), "Les valeurs doivent être comprises entre 0 et 1"

# --- 9. Export en fichier CSV
df_sortie.to_csv(FICHIER_CSV_SORTIE, index=False)
print(f"\n Fichier CSV exporté avec succès dans :\n{FICHIER_CSV_SORTIE}")



--- Contrôles de validation ---
Nombre total de lignes : 2020705
Valeurs NaN : 0
Min : 0.0000 | Max : 1.0000
Moyenne : 0.0020
Mailles avec bâti : 20,495 / 2,020,705 ( 1.0%)

 Fichier CSV exporté avec succès dans :
C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\densite_bati_100m.csv


# Étape 3 (version Casablanca) : `densite_routes`

Champ de la table **Maille** : `densite_routes` (float).

Règle du diagramme : `densite_routes` = Σ longueur de route ∩ maille ÷ surface maille.

La couche des routes ne couvre que Casablanca, alors que les mailles couvrent toute la région Casablanca-Settat.

- **Mailles de Casablanca** : calcul normal, une maille sans route vaut 0 (l'absence de route est une vraie information).
- **Mailles hors Casablanca** : valeur **vide** (NaN), car on ne sait pas s'il y a des routes.

## 1. Imports, chemins et lecture des couches

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

# --- Racine du projet (le notebook peut être lancé depuis /scripts)
ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent

# --- Entrées
DOSSIER_CLEAN = ROOT / "donnees_traitees" / "casablanca_settat_clean"
FICHIER_ROUTES = DOSSIER_CLEAN / "routes_casablanca_settat_reprojete.gpkg"
FICHIER_COMMUNES = DOSSIER_CLEAN / "communes_casablanca_settat_clean.gpkg"

TAILLE_MAILLE = 100                      # 50 pour le test à 50 m
DOSSIER_MAILLES = ROOT / "donnees_traitees" / "mailles"
FICHIER_MAILLES = DOSSIER_MAILLES / f"mailles_{TAILLE_MAILLE}m.gpkg"

# --- Sortie
FICHIER_SORTIE = DOSSIER_MAILLES / f"densite_routes_casablanca_{TAILLE_MAILLE}m.csv"

CRS_PROJET = 26191                       # CRS unique du diagramme

# --- Paramètres de sélection de Casablanca
SEUIL_KM = 5                             # une commune est « Casablanca » si elle contient au moins 5 km de route
COMMUNES_CASA_IDS = None                 # ou une liste d'id fixée à la main, ex. [12, 13, 14]

# --- Mailles : id, taille_m, commune_id (+ géométrie)
mailles = gpd.read_file(
    FICHIER_MAILLES, layer="maille", columns=["id", "taille_m", "commune_id"]
).rename_geometry("geom")

# --- Routes : table Route du diagramme
routes = gpd.read_file(FICHIER_ROUTES).rename_geometry("geom")

# Les deux couches doivent être en EPSG:26191 (longueurs en mètres)
assert mailles.crs.to_epsg() == CRS_PROJET, "Mailles : CRS différent de 26191"
if routes.crs.to_epsg() != CRS_PROJET:
    routes = routes.to_crs(CRS_PROJET)

print(f"Mailles : {len(mailles):,} | Routes : {len(routes):,}")
print(routes["type_voie"].value_counts(dropna=False))

Mailles : 2,020,705 | Routes : 4,290
type_voie
secondary    1836
primary      1544
trunk         770
motorway      140
Name: count, dtype: int64


## 2. Découpe des routes par les mailles

Chaque morceau de route garde l'`id` et le `commune_id` de la maille qu'il traverse.

In [3]:
# Mailles touchées par au moins une route
idx_mailles = gpd.sjoin(
    mailles[["id", "geom"]], routes[["geom"]], how="inner", predicate="intersects"
).index.unique()
mailles_touchees = mailles.loc[idx_mailles, ["id", "commune_id", "geom"]]

# Découpe : un morceau de route par maille traversée
morceaux = gpd.overlay(
    routes[["geom"]], mailles_touchees, how="intersection", keep_geom_type=True
)
morceaux["longueur_m"] = morceaux.geometry.length

print(f"Mailles traversées : {len(mailles_touchees):,} | Morceaux : {len(morceaux):,}")

Mailles traversées : 8,906 | Morceaux : 20,165


## 3. Délimiter Casablanca : communes qui contiennent des routes

On calcule la longueur de route par commune. Les communes avec au moins `SEUIL_KM` km de route forment la zone « Casablanca ». **Vérifiez la liste affichée** : si une commune est mal classée, fixez `COMMUNES_CASA_IDS` à la main.

In [4]:
# Longueur de route par commune (km)
long_commune = (
    morceaux.groupby("commune_id")["longueur_m"].sum().div(1000)
    .rename("longueur_km").reset_index()
)

# Noms des communes (si la couche contient une colonne "nom")
communes = gpd.read_file(FICHIER_COMMUNES)
if "id" not in communes.columns:
    communes["id"] = np.arange(1, len(communes) + 1)     # même règle qu'à l'étape 1
if "nom" in communes.columns:
    long_commune = long_commune.merge(
        communes[["id", "nom"]], left_on="commune_id", right_on="id", how="left"
    ).drop(columns="id")

long_commune = long_commune.sort_values("longueur_km", ascending=False)
print(long_commune.to_string(index=False))

# Sélection des communes de Casablanca
if COMMUNES_CASA_IDS is None:
    COMMUNES_CASA_IDS = long_commune.loc[
        long_commune["longueur_km"] >= SEUIL_KM, "commune_id"
    ].astype(int).tolist()

print(f"\nCommunes retenues pour Casablanca : {len(COMMUNES_CASA_IDS)}")

 commune_id  longueur_km                                                       nom
         23   645.080337                  Casablanca ⵜⴰⴷⴷⴰⵔⵜ ⵜⵓⵎⵍⵉⵍⵜ الدار البيضاء
         21    93.063193                                         Bouskoura بوسكورة
         76    75.651982                                       Mohammédia المحمدية
         34    69.788578               El Majjatia Ouled Taleb المجاطية أولاد طالب
          5    64.041243                                    Aïn Harrouda عين حرودة
        136    43.441149 Sidi Hajjaj Oued Hassar ⵙⵉⴷⵉ ⵃⵊⵊⴰⵊ ⵡⴰⴷ ⵃⵚⵚⵕ سيدي حجاج واد
         91    40.832122                        Oulad Azzouz ⵡⵍⴰⴷ ⵄⵣⵣⵓⵣ أولاد عزوز
         28    40.024283                          Dar Bouazza ⴷⴰⵕ ⴱⵓⵄⵣⵣⴰ دار بوعزة
         25    37.740216                               Chellalate ⵛⵍⵍⴰⵍⴰⵜ الشلالات
         57    25.396679                                     Lahraouyine الهراويين
         17    17.035037                             Bni Yakhlef ⴱⵏⵉ ⵉⵅⵍⴼ بني يخلف
    

## 4. Calcul de `densite_routes`

- Mailles de Casablanca : Σ longueurs ÷ `taille_m ** 2`, une maille sans route vaut 0
- Mailles hors Casablanca : valeur vide (NaN)

In [5]:
# Mailles appartenant à Casablanca (commune du centroïde, règle du diagramme)
mailles["en_casa"] = mailles["commune_id"].isin(COMMUNES_CASA_IDS)

# Somme des longueurs par maille
longueur_par_maille = morceaux.groupby("id")["longueur_m"].sum()
mailles["longueur_routes_m"] = mailles["id"].map(longueur_par_maille).fillna(0.0)

# Densité = longueur ÷ surface de la maille (taille_m ** 2) ; vide hors Casablanca
mailles["densite_routes"] = (
    mailles["longueur_routes_m"] / (mailles["taille_m"] ** 2)
).where(mailles["en_casa"])

print(f"Mailles de Casablanca : {mailles['en_casa'].sum():,}")
print(f"Mailles hors Casablanca (valeur vide) : {mailles['densite_routes'].isna().sum():,}")

Mailles de Casablanca : 125,771
Mailles hors Casablanca (valeur vide) : 1,894,934


## 5. Contrôles

- Densité positive ou nulle
- Aucune valeur vide dans Casablanca
- Part de la longueur de routes retenue dans la zone Casablanca

In [6]:
casa = mailles[mailles["en_casa"]]

assert (casa["densite_routes"] >= 0).all(), "Densité négative"
assert casa["densite_routes"].notna().all(), "Valeur vide dans Casablanca"
assert mailles.loc[~mailles["en_casa"], "densite_routes"].isna().all()

l_total = morceaux["longueur_m"].sum()
l_casa = casa["longueur_routes_m"].sum()
print(f"Longueur découpée (toutes mailles) : {l_total / 1000:,.1f} km")
print(f"Longueur dans la zone Casablanca   : {l_casa / 1000:,.1f} km ({l_casa / l_total:.1%})")
print(f"Densité maximale : {casa['densite_routes'].max():.4f} m/m²")

Longueur découpée (toutes mailles) : 1,232.2 km
Longueur dans la zone Casablanca   : 1,220.9 km (99.1%)
Densité maximale : 0.0646 m/m²


## 6. Test : garder `densite_routes` comme critère ?

Le test porte uniquement sur les mailles de Casablanca, où les routes sont disponibles.

In [7]:
d = casa["densite_routes"]
non_nul = d[d > 0]

print(f"Mailles avec route : {len(non_nul):,} / {len(d):,} ({len(non_nul) / len(d):.1%})")
print(f"Mailles sans route : {(d == 0).sum():,} ({(d == 0).mean():.1%})")
print("\nDistribution des valeurs non nulles :")
print(non_nul.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]))
print(f"\nÉcart-type / moyenne (mailles de Casablanca) : {d.std() / d.mean():.2f}")

Mailles avec route : 8,803 / 125,771 (7.0%)
Mailles sans route : 116,968 (93.0%)

Distribution des valeurs non nulles :
count    8803.000000
mean        0.013869
std         0.009124
min         0.000001
25%         0.006709
50%         0.012353
75%         0.020507
95%         0.026299
99%         0.043245
max         0.064620
Name: densite_routes, dtype: float64

Écart-type / moyenne (mailles de Casablanca) : 4.41


**Décision à noter dans le rapport :**
- la variable n'est calculée que pour Casablanca ; elle reste vide ailleurs ;
- elle ne peut devenir un critère que si l'analyse est limitée à Casablanca (une maille sans valeur ne peut pas être regroupée) ;
- les routes sont uniquement primaires et secondaires : la densité est sous-estimée là où il y a surtout des petites rues.

## 7. Sauvegarde

Seul le champ du diagramme (`id`, `densite_routes`) est enregistré. Les mailles hors Casablanca ont une valeur vide. La jointure avec `Maille` se fait sur `id`.

In [8]:
mailles[["id", "densite_routes"]].to_csv(FICHIER_SORTIE, index=False)
print("Fichier écrit :", FICHIER_SORTIE)

Fichier écrit : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\densite_routes_casablanca_100m.csv


In [10]:
non_vides = mailles[mailles["densite_routes"].notna()]
print(f"Mailles non vides : {len(non_vides):,}")
non_vides[["id", "commune_id", "densite_routes"]].head(20)

Mailles non vides : 125,771


,id,commune_id,densite_routes
1567444,1571363,115,0.0
1567445,1571364,115,0.0
1567446,1571365,115,0.0
1568893,1572814,115,0.0
1568894,1572815,115,0.0
1568895,1572816,115,0.0
1568896,1572817,115,0.0
1568897,1572818,115,0.0
1568898,1572819,115,0.0
1568899,1572820,115,0.0


## Étape 4 : pct_impermeable et ruissel_moy (overlay avec union)
Consignes respectées :
1. Élimination des chevauchements internes d'OccupationSol.
2. pct_impermeable : Union(sol impermeable, bâti, emprise routes lanes*3.5m / 7m).
3. ruissel_moy : Moyenne de coef_ruissel pondérée par la surface des morceaux.
4. Zones blanches (non couvertes) : 0% pour l'imperméable, 0.20 pour le ruissellement.
5. Sortie CSV : donnees_traitees/mailles/impermeable_ruissellement_100m.csv

In [31]:
import shapely

# --- 1. Chemins et Configuration
FICHIER_MAILLES = DOSSIER_MAILLES / "mailles_100m.gpkg"
FICHIER_OCSOL = DOSSIER_CLEAN / "occupation_sol_casablanca_settat_clean.gpkg"
FICHIER_BATI = DOSSIER_CLEAN / "bati_casablanca_settat_clean.gpkg"
FICHIER_ROUTES = DOSSIER_CLEAN / "routes_casablanca_settat_reprojete.gpkg"
FICHIER_CSV_SORTIE = DOSSIER_MAILLES / "impermeable_ruissellement_100m.csv"
print("Lecture des données...")
mailles = gpd.read_file(FICHIER_MAILLES, layer="maille")
if "geometry" in mailles.columns and "geom" not in mailles.columns:
  mailles = mailles.rename_geometry("geom")
elif "geom" in mailles.columns and mailles.geometry.name != "geom":
  mailles = mailles.set_geometry("geom")

ocsol = gpd.read_file(FICHIER_OCSOL)
if ocsol.crs.to_epsg() != CRS_PROJET:
  ocsol = ocsol.to_crs(CRS_PROJET)

bati = gpd.read_file(FICHIER_BATI)
if bati.crs.to_epsg() != CRS_PROJET:
  bati = bati.to_crs(CRS_PROJET)

routes = gpd.read_file(FICHIER_ROUTES)
if routes.crs.to_epsg() != CRS_PROJET:
  routes = routes.to_crs(CRS_PROJET)

print(f"Mailles : {len(mailles):,} | OccupationSol : {len(ocsol):,}")
print(f"Bâtiments : {len(bati):,} | Routes : {len(routes):,}")

# Bounding box filter
bbox = mailles.total_bounds
ocsol = ocsol.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]].copy()
bati = bati.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]].copy()
routes = routes.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]].copy()

mailles_geom = mailles[["id", "geom"]].copy()
surface_maille = mailles["geom"].area  # 10 000 m²


Lecture des données...
Mailles : 2,020,705 | OccupationSol : 11,810
Bâtiments : 84,962 | Routes : 4,290


PARTIE 1 : RUISSELLEMENT MOYEN (ruissel_moy) 

In [32]:
print("\n--- 1. Calcul du ruissellement moyen (ruissel_moy) ---")
ocsol_valid = ocsol[ocsol.geometry.is_valid & ~ocsol.geometry.is_empty][
    ["landuse_class", "coef_ruissel", "geometry"]
].copy()

# Intersection vectorielle
morceaux_ocsol = gpd.overlay(
    ocsol_valid, mailles_geom, how="intersection", keep_geom_type=True
)
morceaux_ocsol["surf_m2"] = morceaux_ocsol.geometry.area
morceaux_ocsol["prod_ruissel"] = (
    morceaux_ocsol["surf_m2"] * morceaux_ocsol["coef_ruissel"]
)

# Agrégation rapide par maille
somme_prod = morceaux_ocsol.groupby("id")["prod_ruissel"].sum()
somme_surf_ocsol = morceaux_ocsol.groupby("id")["surf_m2"].sum()

COEF_DEFAUT = 0.20
surf_couverte = mailles["id"].map(somme_surf_ocsol).fillna(0.0)
prod_couvert = mailles["id"].map(somme_prod).fillna(0.0)

surf_non_couverte = (surface_maille - surf_couverte).clip(lower=0.0)
ruissel_moy = (prod_couvert + (surf_non_couverte * COEF_DEFAUT)) / surface_maille
ruissel_moy = ruissel_moy.clip(lower=0.0, upper=1.0)
print("[OK] ruissel_moy calculé.")


--- 1. Calcul du ruissellement moyen (ruissel_moy) ---
[OK] ruissel_moy calculé.


PARTIE 2 : POURCENTAGE IMPERMÉABLE (pct_impermeable)

In [33]:
print("\n--- 2. Calcul de l'imperméabilité (pct_impermeable) ---")

# A. Emprise routes (bufferisé par vecteur numpy rapide)
def extraire_largeurs(df_routes):
  # Vectorisation rapide des largeurs
  lanes = pd.to_numeric(df_routes.get("lanes", 2), errors="coerce").fillna(2.0)
  lanes = np.where(lanes <= 0, 2.0, lanes)
  return (lanes * 3.5) / 2.0  # rayon


rayons = extraire_largeurs(routes)
emprise_routes = routes.geometry.buffer(rayons)
gdf_routes_poly = gpd.GeoDataFrame(
    {"geometry": emprise_routes[~emprise_routes.is_empty]}, crs=CRS_PROJET
)

# B. Sol imperméable
sol_imper = ocsol_valid[ocsol_valid["landuse_class"] == "impermeable"][
    ["geometry"]
].copy()
bati_poly = bati[["geometry"]].copy()

# C. Concaténation des 3 couches EN LIGNES (pas d'union globale géante !)
print("Concaténation des couches imperméables...")
couches_imper = pd.concat(
    [sol_imper, bati_poly, gdf_routes_poly], ignore_index=True
)
couches_imper = gpd.GeoDataFrame(
    couches_imper, geometry="geometry", crs=CRS_PROJET
)

# D. Filtre spatial : on n'intersecte QUE les mailles qui touchent l'imperméable
print("Filtrage spatial préliminaire (Index spatial R-Tree)...")
mailles_candidates_idx = mailles_geom.sjoin(
    couches_imper, how="inner", predicate="intersects"
).index.unique()

mailles_actives = mailles_geom.loc[mailles_candidates_idx]
print(
    f"Mailles concernées : {len(mailles_actives):,} / {len(mailles):,} (les"
    " autres sont à 0%)"
)

# E. Découpage par les mailles actives
print("Découpage géométrique ciblé...")
morceaux_imper = gpd.overlay(
    couches_imper, mailles_actives, how="intersection", keep_geom_type=True
)

# F. UNION LOCALE PAR MAILLE (Magie de l'optimisation)
# Au lieu d'unir 500 000 polygones ensemble, on unit les quelques morceaux à l'intérieur de chaque maille
print(
    "Union locale par maille pour éliminer les chevauchements sans ralentir..."
)
# Regroupe les géométries de chaque maille et fait l'union locale
union_par_maille = morceaux_imper.groupby("id")["geometry"].apply(
    shapely.union_all
)

# G. Calcul de la surface réelle non doublée
surf_imper_par_maille = union_par_maille.geometry.area

# H. Rapprochement avec toutes les mailles (celles non touchées restent à 0.0)
surf_imper_totale = mailles["id"].map(surf_imper_par_maille).fillna(0.0)
pct_impermeable = (surf_imper_totale / surface_maille) * 100.0
pct_impermeable = pct_impermeable.clip(lower=0.0, upper=100.0)
print("[OK] pct_impermeable calculé.")


--- 2. Calcul de l'imperméabilité (pct_impermeable) ---
Concaténation des couches imperméables...
Filtrage spatial préliminaire (Index spatial R-Tree)...
Mailles concernées : 64,332 / 2,020,705 (les autres sont à 0%)
Découpage géométrique ciblé...
Union locale par maille pour éliminer les chevauchements sans ralentir...
[OK] pct_impermeable calculé.


PARTIE 3 : FORMATAGE ET EXPORT CSV

In [34]:

print("\n--- Assemblage du DataFrame final ---")
df_sortie = pd.DataFrame({
    "id": mailles["id"].astype(int),
    "pct_impermeable": pct_impermeable.round(4),
    "ruissel_moy": ruissel_moy.round(4),
})

# Assertions de contrôle
assert (
    df_sortie["pct_impermeable"].isna().sum() == 0
), "NaN dans pct_impermeable"
assert df_sortie["ruissel_moy"].isna().sum() == 0, "NaN dans ruissel_moy"

# Export CSV
df_sortie.to_csv(FICHIER_CSV_SORTIE, index=False)
print(f"\n[SUCCÈS] Fichier CSV exporté dans :\n{FICHIER_CSV_SORTIE}")



--- Assemblage du DataFrame final ---

[SUCCÈS] Fichier CSV exporté dans :
C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\impermeable_ruissellement_100m.csv


## Étape 5 : indice_equip (jointure spatiale Points dans Polygones)
Consignes respectées :
1. Jointure spatiale : chaque POI est rattaché à la maille qui le contient (within).
2. Additionne les poids des POI par maille.
3. Une maille sans POI vaut 0.
4. Sortie CSV : donnees_traitees/mailles/indice_equip_100m.csv

In [35]:
# --- 1. Chemins et Configuration
FICHIER_MAILLES = DOSSIER_MAILLES / "mailles_100m.gpkg"
FICHIER_POIS = DOSSIER_CLEAN / "equipements_poi_casablanca_settat_clean.gpkg"
FICHIER_CSV_SORTIE = DOSSIER_MAILLES / "indice_equip_100m.csv"


print("Lecture des données...")

# Lecture des mailles
mailles = gpd.read_file(FICHIER_MAILLES, layer="maille")
if "geometry" in mailles.columns and "geom" not in mailles.columns:
  mailles = mailles.rename_geometry("geom")
elif "geom" in mailles.columns and mailles.geometry.name != "geom":
  mailles = mailles.set_geometry("geom")

# Lecture des POIs
pois = gpd.read_file(FICHIER_POIS)
if pois.crs.to_epsg() != CRS_PROJET:
  pois = pois.to_crs(CRS_PROJET)

print(f"Nombre de mailles : {len(mailles):,}")
print(f"Nombre de POIs : {len(pois):,}")

# Vérification de la présence de la colonne 'poids'
if "poids" not in pois.columns:
  print("⚠️ Colonne 'poids' absente dans POIs : attribution d'un poids de 1 par défaut.")
  pois["poids"] = 1
else:
  # Remplacement d'éventuels NaN par un poids neutre de 1
  pois["poids"] = pois["poids"].fillna(1).astype(int)

# --- 2. Jointure spatiale (Points dans Polygones)
print("Exécution de la jointure spatiale (Points within Mailles)...")
# On ne garde que les colonnes nécessaires pour accélérer la jointure
pois_sub = pois[["poids", "geometry"]].copy()
mailles_sub = mailles[["id", "geom"]].copy()

# Chaque point reçoit l'id de la maille qui le contient
jointure_pois = gpd.sjoin(
    pois_sub,
    mailles_sub,
    how="inner",
    predicate="within"
)

# --- 3. Somme des poids des POI par maille
print("Calcul de la somme des poids par maille...")
somme_poids_par_maille = jointure_pois.groupby("id")["poids"].sum()

# --- 4. Rapprochement avec l'ensemble des mailles
# Règle : une maille sans POI vaut 0
indice_equip = mailles["id"].map(somme_poids_par_maille).fillna(0).astype(int)

# --- 5. Création du DataFrame final
df_sortie = pd.DataFrame({
    "id": mailles["id"].astype(int),
    "indice_equip": indice_equip
})

# --- 6. Contrôles et assertions métier
print("\n--- Contrôles de validation ---")
print("Total mailles traitées :", len(df_sortie))
print("Valeurs NaN :", df_sortie["indice_equip"].isna().sum())
print(f"Min : {df_sortie['indice_equip'].min()} | Max : {df_sortie['indice_equip'].max()}")
print(f"Moyenne : {df_sortie['indice_equip'].mean():.4f}")
print(f"Mailles contenant au moins un POI : {(df_sortie['indice_equip'] > 0).sum():,} / {len(df_sortie):,} ({(df_sortie['indice_equip'] > 0).mean() * 100:.2f}%)")

assert df_sortie["indice_equip"].isna().sum() == 0, "Il ne doit pas y avoir de NaN"
assert (df_sortie["indice_equip"] >= 0).all(), "L'indice d'équipement doit être positif ou nul"

# --- 7. Export en fichier CSV
df_sortie.to_csv(FICHIER_CSV_SORTIE, index=False)
print(f"\n[OK] Fichier CSV exporté avec succès dans :\n{FICHIER_CSV_SORTIE}")

# Aperçu
display(df_sortie.head(10))

Lecture des données...
Nombre de mailles : 2,020,705
Nombre de POIs : 2,709
Exécution de la jointure spatiale (Points within Mailles)...
Calcul de la somme des poids par maille...

--- Contrôles de validation ---
Total mailles traitées : 2020705
Valeurs NaN : 0
Min : 0 | Max : 27
Moyenne : 0.0053
Mailles contenant au moins un POI : 2,353 / 2,020,705 (0.12%)

[OK] Fichier CSV exporté avec succès dans :
C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\indice_equip_100m.csv


,id,indice_equip
0,2,0
1,6,0
2,7,0
3,8,0
4,10,0
5,11,0
6,12,0
7,13,0
8,14,0
9,17,0


## Étape 6 : distance_secours (Analyse Réseau Routier)

Consignes respectées :
1. Nodage des intersections routières (shapely.node).
2. Extraction de la plus grande composante connexe du réseau (élimination des tronçons isolés).
3. Projection des centroïdes et des casernes sur les nœuds les plus proches (cKDTree).
4. Calcul du plus court chemin (Dijkstra multi-sources) vers la caserne la plus proche.
5. Sortie CSV : donnees_traitees/mailles/distance_secours_100m.csv


In [36]:
import networkx as nx
from scipy.spatial import cKDTree
import shapely

# --- 1. Chemins et Configuration
FICHIER_ROUTES = (
    DOSSIER_CLEAN / "routes_casablanca_settat_reprojete.gpkg"
)  
FICHIER_SECOURS = (
    DOSSIER_CLEAN / "infrastructures_secours_clean.gpkg"
)
FICHIER_CSV_SORTIE = DOSSIER_MAILLES / "distance_secours_100m.csv"
print("Lecture des données...")
# A. Mailles
mailles = gpd.read_file(FICHIER_MAILLES, layer="maille")
if "geometry" in mailles.columns and "geom" not in mailles.columns:
  mailles = mailles.rename_geometry("geom")
elif "geom" in mailles.columns and mailles.geometry.name != "geom":
  mailles = mailles.set_geometry("geom")

# Récupération des centroïdes en géométrie Point
if isinstance(mailles["centroide"].iloc[0], str):
  centroides = gpd.GeoSeries.from_wkt(mailles["centroide"], crs=CRS_PROJET)
else:
  centroides = mailles["geom"].centroid

# B. Routes
routes = gpd.read_file(FICHIER_ROUTES)
if routes.crs.to_epsg() != CRS_PROJET:
  routes = routes.to_crs(CRS_PROJET)

# C. Infrastructures de secours (casernes, protection civile)
secours = gpd.read_file(FICHIER_SECOURS)
if secours.crs.to_epsg() != CRS_PROJET:
  secours = secours.to_crs(CRS_PROJET)

print(f"Mailles : {len(mailles):,} | Tronçons routes : {len(routes):,}")
print(f"Infrastructures de secours : {len(secours)}")

Lecture des données...
Mailles : 2,020,705 | Tronçons routes : 4,290
Infrastructures de secours : 50


PARTIE 1 : TOPOLOGIE ET NODAGE DU RÉSEAU ROUTIER

In [37]:
print("\n--- 1. Nodage et construction du graphe routier ---")

# Filtrer sur les géométries valides et non vides
lignes_routes = routes.geometry[
    routes.geometry.is_valid & ~routes.geometry.is_empty
].values

# Fusion et découpe stricte à chaque intersection (shapely.node)
print("Nodage des intersections (création des carrefours)...")
lignes_unies = shapely.union_all(lignes_routes)
lignes_nodees = shapely.node(lignes_unies)

# Extraction de tous les segments simples (LineString)
if lignes_nodees.geom_type == "MultiLineString":
  segments = list(lignes_nodees.geoms)
elif lignes_nodees.geom_type == "LineString":
  segments = [lignes_nodees]
else:
  # Cas de GeometryCollection résiduel
  segments = [g for g in lignes_nodees.geoms if g.geom_type == "LineString"]

print(f"Nombre de segments nodés : {len(segments):,}")

# Construction du graphe NetworkX
G = nx.Graph()
for seg in segments:
  coords = list(seg.coords)
  for i in range(len(coords) - 1):
    u = (round(coords[i][0], 2), round(coords[i][1], 2))
    v = (round(coords[i + 1][0], 2), round(coords[i + 1][1], 2))
    longueur = float(shapely.Point(u).distance(shapely.Point(v)))
    if longueur > 0:
      G.add_edge(u, v, weight=longueur)

print(
    f"Graphe initial : {G.number_of_nodes():,} nœuds, {G.number_of_edges():,}"
    " arêtes."
)

# Conservation de la plus grande composante connexe (suppression des tronçons isolés)
composantes = list(nx.connected_components(G))
plus_grande_composante = max(composantes, key=len)
G_principal = G.subgraph(plus_grande_composante).copy()
print(
    "Composante principale connectée :"
    f" {G_principal.number_of_nodes():,} nœuds ("
    f" {len(G_principal) / len(G) * 100:.1f}% du réseau conservé)."
)



--- 1. Nodage et construction du graphe routier ---
Nodage des intersections (création des carrefours)...
Nombre de segments nodés : 5,313
Graphe initial : 35,949 nœuds, 37,150 arêtes.
Composante principale connectée : 35,881 nœuds ( 99.8% du réseau conservé).


PARTIE 2 : INDEX SPATIAL DES NŒUDS DU RÉSEAU (KD-Tree)

In [38]:
print("\n--- 2. Indexation spatiale des nœuds routiers ---")
noeuds_coords = np.array(list(G_principal.nodes()))
kdtree = cKDTree(noeuds_coords)


--- 2. Indexation spatiale des nœuds routiers ---


PARTIE 3 : MULTI-SOURCE DIJKSTRA DEPUIS LES CASERNES

In [39]:
print("\n--- 3. Calcul du plus court chemin multi-sources ---")

# Raccordement des casernes au nœud du réseau le plus proche
coords_secours = np.array([[pt.x, pt.y] for pt in secours.geometry])
dist_secours_vers_noeud, idx_noeuds_secours = kdtree.query(coords_secours)

# Dictionnaire des nœuds de départ avec leur pénalité d'accès initiale (distance à vol d'oiseau)
sources_secours = {}
for i, idx in enumerate(idx_noeuds_secours):
  noeud = tuple(noeuds_coords[idx])
  d_approche = dist_secours_vers_noeud[i]
  # Si plusieurs casernes partagent le même nœud, on garde la plus proche
  if noeud not in sources_secours or d_approche < sources_secours[noeud]:
    sources_secours[noeud] = d_approche

print(
    f"{len(secours)} casernes rattachées à {len(sources_secours)} nœuds"
    " d'entrée distincts."
)

# Algorithme de Dijkstra multi-sources en 1 seule passe sur tout le graphe !
# Calcule la distance minimale réseau de chaque nœud du graphe vers la caserne la plus proche
print("Exécution de l'algorithme Dijkstra global...")
distances_reseau_noeuds = nx.multi_source_dijkstra_path_length(
    G_principal, sources=sources_secours.keys(), weight="weight"
)

# Ajout de la distance d'approche pour chaque nœud source
for noeud_source, d_init in sources_secours.items():
  distances_reseau_noeuds[noeud_source] += d_init



--- 3. Calcul du plus court chemin multi-sources ---
50 casernes rattachées à 44 nœuds d'entrée distincts.
Exécution de l'algorithme Dijkstra global...


PARTIE 4 : PROJECTION DES MAILLLES ET DISTANCE TOTALE

In [40]:
print("\n--- 4. Calcul de la distance d'accès pour chaque maille ---")
coords_mailles = np.array([[pt.x, pt.y] for pt in centroides])

# Recherche du nœud routier le plus proche pour chaque centroïde
dist_maille_vers_noeud, idx_noeuds_mailles = kdtree.query(
    coords_mailles, workers=-1
)

# Vecteur de distance réseau du nœud de rattachement
noeuds_mailles_tuples = [
    tuple(noeuds_coords[idx]) for idx in idx_noeuds_mailles
]
dist_reseau_noeud = np.array(
    [distances_reseau_noeuds.get(n, np.nan) for n in noeuds_mailles_tuples]
)

# Distance totale = Distance centroïde->nœud routier + Distance réseau nœud->caserne
dist_totale_secours = dist_maille_vers_noeud + dist_reseau_noeud


--- 4. Calcul de la distance d'accès pour chaque maille ---


PARTIE 5 : CONTRÔLES ET EXPORT CSV

In [41]:
print("\n--- 5. Formatage et Export CSV ---")
df_sortie = pd.DataFrame({
    "id": mailles["id"].astype(int),
    "distance_secours": np.round(dist_totale_secours, 2),
})

# Remplacement de rares cas non connectés par la médiane ou distance directe
if df_sortie["distance_secours"].isna().sum() > 0:
  print(
      f"⚠️ {df_sortie['distance_secours'].isna().sum()} mailles sans connexion"
      " réseau : imputation par distance euclidienne directe."
  )
  union_secours = shapely.union_all(secours.geometry.values)
  d_directe = centroides.distance(union_secours)
  df_sortie["distance_secours"] = df_sortie["distance_secours"].fillna(
      d_directe
  )

print("\n--- Contrôles de validation ---")
print(f"Total mailles traitées : {len(df_sortie):,}")
print("Valeurs NaN :", df_sortie["distance_secours"].isna().sum())
print(
    f"Distance min  : {df_sortie['distance_secours'].min():.2f} m ("
    f" {df_sortie['distance_secours'].min() / 1000:.2f} km)"
)
print(
    f"Distance max  : {df_sortie['distance_secours'].max():.2f} m ("
    f" {df_sortie['distance_secours'].max() / 1000:.2f} km)"
)
print(
    f"Distance moy. : {df_sortie['distance_secours'].mean():.2f} m ("
    f" {df_sortie['distance_secours'].mean() / 1000:.2f} km)"
)

assert (
    df_sortie["distance_secours"].isna().sum() == 0
), "NaN détectés dans distance_secours"
assert (
    df_sortie["distance_secours"] >= 0
).all(), "Les distances doivent être positives"

# Export CSV
df_sortie.to_csv(FICHIER_CSV_SORTIE, index=False)
print(f"\n[SUCCÈS] Fichier CSV généré dans :\n{FICHIER_CSV_SORTIE}")

# Aperçu
display(df_sortie.head(10))


--- 5. Formatage et Export CSV ---

--- Contrôles de validation ---
Total mailles traitées : 2,020,705
Valeurs NaN : 0
Distance min  : 24.19 m ( 0.02 km)
Distance max  : 252798.62 m ( 252.80 km)
Distance moy. : 91194.55 m ( 91.19 km)

[SUCCÈS] Fichier CSV généré dans :
C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\distance_secours_100m.csv


,id,distance_secours
0,2,139442.57
1,6,139332.43
2,7,139343.15
3,8,139353.96
4,10,139211.81
5,11,139222.36
6,12,139233.00
7,13,139243.73
8,14,139254.55
9,17,139101.90


# Étape 7 : `distance_oued`

Champ de la table **Maille** : `distance_oued` (float, en mètres).

Règle du diagramme : `distance_oued` = distance du centroïde de la maille à l'objet `CoursEau` le plus proche.

1. Vérifier le contenu de la couche des cours d'eau (vrais oueds présents ?)
2. Calculer la distance du centroïde au cours d'eau le plus proche
3. Zones tampons (50, 100, 200 m) : contrôle des résultats
4. Test de sensibilité : tous les cours d'eau ou seulement les cours naturels

**Les largeurs 50, 100 et 200 m sont des paramètres de l'étude, pas des valeurs légales.** La loi 36-15 traite de la gestion des risques liés à l'eau, mais les largeurs réglementaires exactes n'ont pas été confirmées. Elles sont à demander à l'agence de bassin hydraulique ou à chercher dans les textes d'application.

## 1. Imports, chemins et lecture des couches

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import rasterio
from rasterio.features import rasterize
from scipy import ndimage
from skimage.graph import MCP_Geometric      # si absent : %pip install scikit-image
# --- Racine du projet (le notebook peut être lancé depuis /scripts)
ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent


# --- MNT et paramètres de pénalité terrain (hypothèses d'étude, à justifier / tester)
DEM_CORR = DOSSIER_CLEAN / "DEM_Grand_Casablanca_corrige.tif"
PENTE_REF_DEG = 5.0      # au-delà, la pente freine fortement la "distance"
DZ_REF_M = 10.0          # dénivelé au-dessus du chenal : +10 m => distance x2
FRICTION_MAX = 100.0     # plafond pour éviter des distances aberrantes
# --- Entrées
DOSSIER_CLEAN = ROOT / "donnees_traitees" / "casablanca_settat_clean"
# ⚠️ Adaptez le nom si votre fichier de cours d'eau s'appelle autrement
FICHIER_COURS_EAU = DOSSIER_CLEAN / "gis_osm_waterways_casablanca_settat_clean.gpkg"

TAILLE_MAILLE = 100                      # 50 pour le test à 50 m
DOSSIER_MAILLES = ROOT / "donnees_traitees" / "mailles"
FICHIER_MAILLES = DOSSIER_MAILLES / f"mailles_{TAILLE_MAILLE}m.gpkg"

# --- Sortie
FICHIER_SORTIE = DOSSIER_MAILLES / f"distance_oued_{TAILLE_MAILLE}m.csv"

CRS_PROJET = 26191                       # CRS unique du diagramme

# --- Paramètres de l'étude (PAS des valeurs légales)
LARGEURS_TAMPON_M = [50, 100, 200]       # bandes autour des cours d'eau
TYPE_TRACE_RETENU = None                 # None = tous les cours d'eau ; "naturel_presume" = river + stream

# --- Mailles : id, taille_m, centroide (WKT, enregistré à l'étape 1) + géométrie
mailles = gpd.read_file(
    FICHIER_MAILLES, layer="maille", columns=["id", "taille_m", "centroide"]
).rename_geometry("geom")

# --- Cours d'eau : table CoursEau du diagramme
cours_eau = gpd.read_file(FICHIER_COURS_EAU).rename_geometry("geom")

# Les deux couches doivent être en EPSG:26191 (distances en mètres)
assert mailles.crs.to_epsg() == CRS_PROJET, "Mailles : CRS différent de 26191"
if cours_eau.crs.to_epsg() != CRS_PROJET:
    cours_eau = cours_eau.to_crs(CRS_PROJET)

print(f"Mailles : {len(mailles):,} | Tronçons de cours d'eau : {len(cours_eau):,}")
print("Colonnes CoursEau :", list(cours_eau.columns))

Mailles : 2,020,705 | Tronçons de cours d'eau : 1,449
Colonnes CoursEau : ['uid', 'uid_parent', 'osm_id', 'fclass', 'nom', 'type_trace', 'len_m', 'geom']


## 2. Vérifier le contenu de la couche `CoursEau`

On vérifie que les vrais oueds sont présents : classes `river` et `stream` (naturels) contre `canal` et `drain` (artificiels), longueurs par classe, et noms des principaux oueds.

In [6]:
# Nombre de tronçons par classe et par type de tracé
print(cours_eau["fclass"].value_counts(dropna=False))
print()
print(cours_eau["type_trace"].value_counts(dropna=False))

# Longueur par classe (km), calculée sur la géométrie
cours_eau["longueur_km"] = cours_eau.geometry.length / 1000
print()
print(cours_eau.groupby("fclass")["longueur_km"].agg(["count", "sum"]).round(1)
      .rename(columns={"count": "troncons", "sum": "longueur_km"}))

# Cohérence entre len_m (champ du diagramme) et la longueur géométrique
ecart_max = (cours_eau["len_m"] - cours_eau.geometry.length).abs().max()
print(f"\nÉcart maximal len_m / longueur géométrique : {ecart_max:.3f} m")

# Oueds nommés (le champ nom est souvent vide)
print(f"\nTronçons avec un nom : {cours_eau['nom'].notna().sum():,} / {len(cours_eau):,}")
print(cours_eau["nom"].value_counts().head(15))

fclass
river     1075
stream     184
canal      150
drain       40
Name: count, dtype: int64

type_trace
naturel_presume       1259
artificiel_presume     190
Name: count, dtype: int64

        troncons  longueur_km
fclass                       
canal        150        373.8
drain         40         77.1
river       1075        679.4
stream       184        481.0

Écart maximal len_m / longueur géométrique : 0.000 m

Tronçons avec un nom : 1,070 / 1,449
nom
Oued Oum Rbia     555
Oum Rabia         242
Oued Felfel       112
Oued Cherrat       52
Ain El Caïd        14
Oued Bouskoura     14
Arrimène           12
Oued Chrouaa       10
Oued Houara        10
Oued Dhaybia        9
Oued Mellah         7
Ain El Isaoui       6
Oued Neguiouat      4
Oued Nefifikh       4
Oued Fertouaou      4
Name: count, dtype: int64


## 3. Distance du centroïde au cours d'eau le plus proche

Le centroïde est lu depuis le champ `centroide` (WKT) de la table `Maille`. La recherche du plus proche voisin utilise un index spatial (`STRtree`), ce qui est rapide même pour plus de 2 millions de mailles.

In [7]:
# Sélection des cours d'eau pris en compte
if TYPE_TRACE_RETENU is None:
    cours_sel = cours_eau
else:
    cours_sel = cours_eau[cours_eau["type_trace"] == TYPE_TRACE_RETENU]
print(f"Tronçons utilisés : {len(cours_sel):,}")

# Centroïdes de toutes les mailles (champ centroide de Maille)
centroides = shapely.from_wkt(mailles["centroide"].values)

def distance_au_plus_proche(points, lignes):
    """Distance (m) de chaque point à la ligne la plus proche."""
    arbre = shapely.STRtree(lignes.geometry.values)
    idx, dist = arbre.query_nearest(points, return_distance=True, all_matches=False)
    resultat = np.full(len(points), np.nan)
    resultat[idx[0]] = dist
    return resultat

mailles["distance_oued"] = distance_au_plus_proche(centroides, cours_sel)

print(mailles["distance_oued"].describe())

Tronçons utilisés : 1,449
count    2.020705e+06
mean     8.702855e+03
std      8.083238e+03
min      1.278215e-03
25%      2.194088e+03
50%      6.100871e+03
75%      1.313985e+04
max      3.838608e+04
Name: distance_oued, dtype: float64


In [8]:
# --- 1. MNT
with rasterio.open(DEM_CORR) as src:
    dem = src.read(1).astype("float64")
    T, res, nd = src.transform, src.res[0], src.nodata
    assert src.crs.to_epsg() == CRS_PROJET or "merchich" in src.crs.to_wkt().lower(), \
        "MNT pas en EPSG:26191"
valid = np.isfinite(dem) & (dem != -9999.0) & ((dem != nd) if nd is not None else True)
dem[~valid] = np.nan
H, W = dem.shape

# --- 2. Pente (degrés)
gy, gx = np.gradient(dem, res)
pente = np.degrees(np.arctan(np.hypot(gx, gy)))
pente = np.nan_to_num(pente, nan=0.0)

# --- 3. Chenaux rastérisés sur la grille du MNT
canal = rasterize(
    [(g, 1) for g in cours_sel.geometry if g is not None and not g.is_empty],
    out_shape=dem.shape, transform=T, fill=0, all_touched=True, dtype="uint8",
).astype(bool) & valid
print(f"Pixels de chenal : {canal.sum():,}")

# --- 4. Dénivelé au-dessus du chenal le plus proche (montagne à côté de l'oued)
idx = ndimage.distance_transform_edt(~canal, return_distances=False, return_indices=True)
dz = np.nan_to_num(dem - dem[idx[0], idx[1]], nan=0.0)
dz = np.clip(dz, 0, None)                      # seul le terrain PLUS HAUT que l'oued pénalise

# --- 5. Surface de friction puis distance de coût (Dijkstra géométrique 8 voisins)
friction = (1 + (pente / PENTE_REF_DEG) ** 2) * (1 + dz / DZ_REF_M)
friction = np.clip(friction, 1.0, FRICTION_MAX)
friction[~valid] = np.inf                      # mer / hors MNT : infranchissable

mcp = MCP_Geometric(friction, fully_connected=True)
cout, _ = mcp.find_costs(list(zip(*np.where(canal))))
cout = cout * res                              # en "mètres équivalents"

# --- 6. Lecture aux centroïdes
x, y = shapely.get_x(centroides), shapely.get_y(centroides)
cols, rows = ~T * (x, y)
rows, cols = np.floor(rows).astype(int), np.floor(cols).astype(int)
hors = (rows < 0) | (rows >= H) | (cols < 0) | (cols >= W)
c = cout[np.clip(rows, 0, H - 1), np.clip(cols, 0, W - 1)]
c[hors] = np.nan

d_eucl = mailles["distance_oued"].values
# jamais inférieur à l'euclidien ; repli sur l'euclidien hors MNT / mer
mailles["distance_oued_mnt"] = np.where(np.isfinite(c), np.maximum(c, d_eucl), d_eucl)

# --- 7. Contrôles
r = mailles["distance_oued_mnt"] / mailles["distance_oued"].replace(0, np.nan)
print(mailles[["distance_oued", "distance_oued_mnt"]].describe().round(1))
print(f"Mailles hors MNT (repli euclidien) : {(~np.isfinite(c)).sum():,}")
print(f"Mailles dont la distance double ou plus : {(r >= 2).sum():,} ({(r >= 2).mean():.1%})")
assert mailles["distance_oued_mnt"].notna().all()

Pixels de chenal : 63,953
       distance_oued  distance_oued_mnt
count      2020705.0          2020705.0
mean          8702.9            59694.6
std           8083.2            90964.8
min              0.0                0.0
25%           2194.1             6217.9
50%           6100.9            21800.5
75%          13139.8            68343.0
max          38386.1           545119.7
Mailles hors MNT (repli euclidien) : 25,260
Mailles dont la distance double ou plus : 1,271,745 (62.9%)


## 4. Contrôles

- Aucune valeur manquante
- Distances positives ou nulles
- Distribution (une région étendue donne de grandes distances maximales)

In [9]:
d = mailles["distance_oued"]

assert d.notna().all(), "Valeur manquante dans distance_oued"
assert (d >= 0).all(), "Distance négative"

print(f"Distance minimale : {d.min():.1f} m")
print(f"Distance médiane  : {d.median():.1f} m")
print(f"Distance maximale : {d.max() / 1000:.1f} km")
for w in LARGEURS_TAMPON_M:
    print(f"Mailles à moins de {w} m d'un cours d'eau : {(d <= w).sum():,} ({(d <= w).mean():.1%})")

Distance minimale : 0.0 m
Distance médiane  : 6100.9 m
Distance maximale : 38.4 km
Mailles à moins de 50 m d'un cours d'eau : 15,287 (0.8%)
Mailles à moins de 100 m d'un cours d'eau : 30,328 (1.5%)
Mailles à moins de 200 m d'un cours d'eau : 59,519 (2.9%)


## 5. Zones tampons : contrôle des distances

On fabrique des bandes de 50, 100 et 200 m autour des cours d'eau. Un centroïde dans une bande doit avoir `distance_oued` inférieure ou égale à la largeur. On ignore une tolérance de 10 cm, car le tampon est une approximation polygonale du cercle.

In [11]:
x = shapely.get_x(centroides)
y = shapely.get_y(centroides)

# Réseau en une seule géométrie, pour le contrôle exact (indépendant du tampon)
lignes_union = shapely.union_all(cours_sel.geometry.values)

bandes = {}
ecarts = {}
for w in LARGEURS_TAMPON_M:
    # Bande de largeur w autour de tous les cours d'eau retenus
    bandes[w] = shapely.union_all(shapely.buffer(cours_sel.geometry.values, w, quad_segs=32))
    dans_bande = shapely.contains_xy(bandes[w], x, y)

    # Centroïdes nettement dedans (d < w - 10 cm) mais hors bande, ou l'inverse
    manque = ~dans_bande & (d.values < w - 0.1)
    en_trop = dans_bande & (d.values > w + 0.1)
    ecarts[w] = np.where(manque | en_trop)[0]

    print(f"Bande {w:>3} m : {dans_bande.sum():,} centroïdes dedans | écarts : {len(ecarts[w])}")

# --- Diagnostic des écarts : qui a raison, le tampon ou la distance ?
for w, indices in ecarts.items():
    for i in indices[:10]:
        pt = shapely.points(x[i], y[i])
        dans = bool(shapely.contains_xy(bandes[w], x[i], y[i]))
        exact = bool(shapely.dwithin(pt, lignes_union, w))   # test exact : distance <= w
        print(
            f"Bande {w} m | maille id={mailles['id'].iloc[i]} | "
            f"distance_oued={d.iloc[i]:.3f} m | dans le tampon={dans} | "
            f"test exact (dwithin)={exact}"
        )

Bande  50 m : 15,287 centroïdes dedans | écarts : 0
Bande 100 m : 30,328 centroïdes dedans | écarts : 0
Bande 200 m : 59,512 centroïdes dedans | écarts : 1
Bande 200 m | maille id=1497415 | distance_oued=199.900 m | dans le tampon=False | test exact (dwithin)=True


## 6. Option : part de la maille dans la bande

Variable supplémentaire (**absente du diagramme, donc non enregistrée dans `Maille`**) : part de la surface de la maille qui tombe dans la bande. Elle sert seulement à contrôler et à cartographier. Désactivée par défaut, car elle est plus lente.

In [9]:
CALCULER_PART_BANDE = False              # mettre True pour l'exécuter

if CALCULER_PART_BANDE:
    parts = {}
    demi_diagonale = TAILLE_MAILLE * np.sqrt(2) / 2
    for w in LARGEURS_TAMPON_M:
        # Seules les mailles proches de la bande peuvent la toucher
        proches = np.where(d.values <= w + demi_diagonale)[0]
        inter = shapely.intersection(mailles["geom"].values[proches], bandes[w])
        part = np.zeros(len(mailles))
        part[proches] = shapely.area(inter) / (TAILLE_MAILLE ** 2)
        parts[w] = part
        print(f"Bande {w} m : {(part > 0).sum():,} mailles touchées, part maximale {part.max():.2f}")

In [10]:
CALCULER_PART_BANDE = True          

if CALCULER_PART_BANDE:
    parts = {}
    demi_diagonale = TAILLE_MAILLE * np.sqrt(2) / 2
    for w in LARGEURS_TAMPON_M:
        # Seules les mailles proches de la bande peuvent la toucher
        proches = np.where(d.values <= w + demi_diagonale)[0]
        inter = shapely.intersection(mailles["geom"].values[proches], bandes[w])
        part = np.zeros(len(mailles))
        part[proches] = shapely.area(inter) / (TAILLE_MAILLE ** 2)
        parts[w] = part
        print(f"Bande {w} m : {(part > 0).sum():,} mailles touchées, part maximale {part.max():.2f}")

Bande 50 m : 34,187 mailles touchées, part maximale 1.00
Bande 100 m : 48,919 mailles touchées, part maximale 1.00
Bande 200 m : 77,582 mailles touchées, part maximale 1.00


## 7. Test de sensibilité : tous les cours d'eau ou seulement les naturels ?

On compare `distance_oued` calculée avec tous les cours d'eau et avec `river` + `stream` seulement. Si peu de mailles changent beaucoup, le choix a peu d'influence.

In [12]:
naturels = cours_eau[cours_eau["type_trace"] == "naturel_presume"]

d_tous = distance_au_plus_proche(centroides, cours_eau)
d_nat = distance_au_plus_proche(centroides, naturels)

ecart = d_nat - d_tous                    # toujours >= 0 (moins de cours d'eau = plus loin)
print(f"Mailles dont la distance change de plus de 50 m  : {(ecart > 50).sum():,} ({(ecart > 50).mean():.1%})")
print(f"Mailles dont la distance change de plus de 200 m : {(ecart > 200).sum():,} ({(ecart > 200).mean():.1%})")
print(f"Corrélation de Spearman : {pd.Series(d_tous).corr(pd.Series(d_nat), method='spearman'):.3f}")

Mailles dont la distance change de plus de 50 m  : 439,755 (21.8%)
Mailles dont la distance change de plus de 200 m : 431,080 (21.3%)
Corrélation de Spearman : 0.915


In [13]:
import pyogrio

# commune_id (lu sans la géométrie : rapide)
communes_mailles = pyogrio.read_dataframe(
    FICHIER_MAILLES, layer="maille", columns=["id", "commune_id"], read_geometry=False
)

comp = pd.DataFrame({
    "id": mailles["id"].values,
    "d_tous": d_tous,
    "d_nat": d_nat,
}).merge(communes_mailles, on="id")
comp["change_50m"] = (comp["d_nat"] - comp["d_tous"]) > 50

# Part de mailles qui changent, par commune (les plus touchées en haut)
par_commune = (
    comp.groupby("commune_id")["change_50m"]
    .agg(mailles="size", changent="sum")
    .assign(part=lambda t: t["changent"] / t["mailles"])
    .sort_values("changent", ascending=False)
)
print(par_commune.head(15))

            mailles  changent      part
commune_id                             
63            25278     25278  1.000000
107           22642     22642  1.000000
113           22231     19982  0.898835
24            18814     17785  0.945307
65            16152     16152  1.000000
109           17233     15474  0.897928
73            14382     13504  0.938951
138           13899     13061  0.939708
135           13552     12563  0.927022
15            12751     12414  0.973571
54            12092     12092  1.000000
23            21606     11458  0.530316
52            15330     11430  0.745597
68            11173     11168  0.999552
146           16898     11087  0.656113


In [14]:
# Noms des communes (même règle d'id qu'à l'étape 1)
FICHIER_COMMUNES = DOSSIER_CLEAN / "communes_casablanca_settat_clean.gpkg"
communes = gpd.read_file(FICHIER_COMMUNES)
if "id" not in communes.columns:
    communes["id"] = np.arange(1, len(communes) + 1)

par_commune_nom = par_commune.reset_index().merge(
    communes[["id", "nom"]], left_on="commune_id", right_on="id", how="left"
).drop(columns="id")

print(par_commune_nom.head(15).to_string(index=False))

# Part de mailles qui changent dans les petites communes (< 5 000 mailles = 50 km²)
petites = par_commune[par_commune["mailles"] < 5000]
print(
    f"\nCommunes < 50 km² : {len(petites)} | mailles qui changent : "
    f"{petites['changent'].sum():,} / {petites['mailles'].sum():,} "
    f"({petites['changent'].sum() / petites['mailles'].sum():.1%})"
)

 commune_id  mailles  changent     part                                      nom
         63    25278     25278 1.000000                 Lgharbia الإثنين الغربية
        107    22642     22642 1.000000                 Oulad Sbaïta أولاد سبيطة
        113    22231     19982 0.898835                  Ouled Ghanem أولاد غانم
         24    18814     17785 0.945307                          Chaïbate شعيبات
         65    16152     16152 1.000000             Loualidia ⵍⵡⴰⵍⵉⴷⵢⵢⴰ الوليدية
        109    17233     15474 0.897928           Oulad Si Bouhya أولاد سي بوحيى
         73    14382     13504 0.938951                             Mettouh متوح
        138    13899     13061 0.939708     Sidi M'Hamed Akhdim سيدي امحمد اخديم
        135    13552     12563 0.927022                 Sidi El Mekki سيدي المكي
         15    12751     12414 0.973571                   Bni Tsiriss بني تسيريس
         54    12092     12092 1.000000                       Laghnadra الغنادرة
         23    21606     114

## 8. Sauvegarde

Seul le champ du diagramme (`id`, `distance_oued`) est enregistré. La jointure avec `Maille` se fait sur `id`.

In [15]:
# distance_oued (champ du diagramme) = distance tenant compte du terrain
out = mailles[["id"]].copy()
out["distance_oued"] = mailles["distance_oued_mnt"].round(2)
out.to_csv(FICHIER_SORTIE, index=False)

# version euclidienne gardée à part (comparaison / annexe du rapport)
mailles[["id", "distance_oued"]].to_csv(
    DOSSIER_MAILLES / f"distance_oued_eucl_{TAILLE_MAILLE}m.csv", index=False
)
print("Fichier écrit :", FICHIER_SORTIE)

Fichier écrit : c:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\distance_oued_100m.csv


# ETAPE 8 – Calculs raster (MNT) et statistiques zonales par maille
**Projet** : vulnérabilité aux inondations urbaines, Casablanca-Settat (EPSG:26191, maille 100 m)

**Champs de la table `Maille` (diagramme de classes) remplis ici** : `pente_moy`, `hand_moy`, `twi_moy`, `flux_max`, `depression_moy` (float).
Règle du diagramme : moyenne zonale du raster (pente, HAND, TWI, dépressions) ; maximum zonal pour le flux.

**Entrées**
- `DEM_Grand_Casablanca_corrige.tif` : MNT 30 m, puits comblés (notebook 03). **Doit être en EPSG:26191.**
- `donnees_traitees/mailles/mailles_100m.gpkg` : table `Maille` de l'étape 1 (couche `maille` : `id`, `geom`, `centroide` (WKT), `taille_m`, `commune_id`)
- `DEM_Grand_Casablanca_30m.tif` (brut) : indispensable pour la profondeur des dépressions (comblé − brut)
- waterways nettoyés (table `CoursEau`) : **la même couche que pour `distance_oued`**, pour densifier le réseau de chenaux du HAND

**Sorties** (aucun fichier .gpkg n'est écrit)
- `donnees_traitees/mailles/variables_mnt_<taille>m.csv` : **uniquement** `id`, `pente_moy`, `hand_moy`, `twi_moy`, `flux_max`, `depression_moy` (jointure avec `Maille` sur `id`)
- `donnees_traitees/mailles/stats_zonales_controle_<taille>m.csv` : colonnes de contrôle hors diagramme (`n_pixels`, `n_pixels_valides`, `depression_moy_brute`, `depression_max_brute`)
- `donnees_traitees/rasters/` : `pente_deg`, `flux_acc`, `hand_m`, `twi`, `depression_m`, `depression_brute_m`, `reseau_chenaux` (GeoTIFF)

**Valeurs vides** : elles sont conservées (jamais remplacées par 0). Elles sont retirées à l'étape 9 : mailles hors MNT (mer), mailles dont le HAND est inconnu (l'eau quitte le MNT sans atteindre un chenal), mailles dont tous les pixels de dépression sont exclus.

**Ordre d'exécution** : 0 (une seule fois, puis redémarrer le noyau) → 1 → 2 → … → 9.



## 0. Installation (une seule fois, puis redémarrer le noyau)


In [ ]:
%pip install pysheds numba geopandas rasterio scipy -q

## 1. Configuration et imports

In [1]:
import os, sys, time
# Neutralise la base PROJ de PostgreSQL (trop ancienne) AVANT de charger rasterio / geopandas
for _v in ("PROJ_LIB", "PROJ_DATA"):
    os.environ.pop(_v, None)

from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
if not hasattr(np, "in1d"):            # compat. pysheds / NumPy 2
    np.in1d = np.isin
from pysheds.grid import Grid
from pysheds.sview import Raster, ViewFinder

# --- Racine du projet (dossier qui contient donnees_traitees/, mailles/, ...)
ROOT = Path(os.environ.get("GEODATA_ROOT", Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd()))
CLEAN = ROOT / "donnees_traitees" / "casablanca_settat_clean"

DEM_CORR  = Path(os.environ.get("DEM_CORR",  CLEAN / "DEM_Grand_Casablanca_corrige.tif"))
DEM_BRUT  = Path(os.environ.get("DEM_BRUT",  ROOT / "donnees_brutes" / "dem" / "DEM_Grand_Casablanca_30m.tif"))
# ⚠️ Doit être la même couche CoursEau que pour distance_oued (étape 5)
WATERWAYS = Path(os.environ.get("WATERWAYS", CLEAN / "gis_osm_waterways_casablanca_settat_clean.gpkg"))
TAILLE_ATTENDUE = int(os.environ.get("TAILLE_MAILLE", 100))      # 100 (ou 50 pour le test) ; vérifiée plus bas dans la table Maille
MAILLES   = Path(os.environ.get("MAILLES",   ROOT / "donnees_traitees" / "mailles" / f"mailles_{TAILLE_ATTENDUE}m.gpkg"))
OUT_R = Path(os.environ.get("OUT_R", ROOT / "donnees_traitees" / "rasters")); OUT_R.mkdir(parents=True, exist_ok=True)

# Recherche automatique des mailles si le chemin ci-dessus est faux
if not MAILLES.exists():
    trouves = sorted(ROOT.rglob(f"mailles_{TAILLE_ATTENDUE}m.gpkg"))
    if not trouves:
        raise FileNotFoundError(f"mailles_{TAILLE_ATTENDUE}m.gpkg introuvable sous {ROOT}. Fixez MAILLES.")
    MAILLES = trouves[0]

# Les tables CSV sont écrites dans le dossier des mailles (à côté de mailles_<taille>m.gpkg)
OUT_T = Path(os.environ.get("OUT_T", MAILLES.parent)); OUT_T.mkdir(parents=True, exist_ok=True)

# --- Paramètres
CRS = "EPSG:26191"
NODATA = -9999.0
SEUIL_RESEAU = 1000      # cellules d'accumulation (≈ 0,9 km²) au-delà desquelles une cellule est un chenal (hypothèse)
TAN_MIN = 0.001          # plancher de tan(pente) pour le TWI
DEPRESSION_MAX_M = 10.0  # seuil d'exclusion de la profondeur par pixel (m) : hypothèse d'étude, au-dessus du 99e centile des moyennes par maille (≈ 7 m)

print("Dossier des tables CSV :", OUT_T)
for nom, p in [("MNT corrigé", DEM_CORR), ("MNT brut", DEM_BRUT), ("Waterways", WATERWAYS), ("Mailles", MAILLES)]:
    print(f"{nom:12s}: {p}  ->  {'OK' if p.exists() else 'ABSENT'}")
assert DEM_CORR.exists() and MAILLES.exists(), "MNT corrigé ou mailles introuvable"
assert WATERWAYS.exists(), "Waterways introuvable : le réseau de chenaux du HAND doit utiliser la même couche CoursEau que distance_oued"

Dossier des tables CSV : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles
MNT corrigé : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\casablanca_settat_clean\DEM_Grand_Casablanca_corrige.tif  ->  OK
MNT brut    : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_brutes\dem\DEM_Grand_Casablanca_30m.tif  ->  OK
Waterways   : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\casablanca_settat_clean\gis_osm_waterways_casablanca_settat_clean.gpkg  ->  OK
Mailles     : C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\mailles\mailles_100m.gpkg  ->  OK


## 2. Lecture du MNT corrigé

In [2]:
with rasterio.open(DEM_CORR) as src:
    dem = src.read(1).astype("float64")
    T = src.transform
    prof = src.profile.copy()
    res = src.res[0]
    crs_mnt = src.crs
    wkt = crs_mnt.to_wkt().lower()
    print("CRS du MNT :", crs_mnt.to_string()[:110], "| EPSG reconnu :", crs_mnt.to_epsg())
    # Le diagramme impose un CRS unique : EPSG:26191 (Merchich / Nord Maroc).
    # Un autre CRS projeté (ex. UTM 29N) décalerait les pixels par rapport aux mailles sans aucune erreur visible.
    assert crs_mnt.to_epsg() == 26191 or ("merchich" in wkt and "nord maroc" in wkt), \
        f"Le MNT n'est pas en EPSG:26191 : {crs_mnt.to_string()[:80]} -> reprojeter le MNT avant de continuer"
    try:
        prof["crs"] = rasterio.crs.CRS.from_epsg(26191)     # étiquette propre pour les rasters écrits
    except Exception as e:
        print("⚠ EPSG:26191 non résolu par PROJ, CRS d'origine conservé :", str(e)[:80])

valid = (dem != NODATA) & np.isfinite(dem)
dem[~valid] = np.nan
print(f"{dem.shape} | pixel {res} m | valides {valid.mean():.0%} | z {np.nanmin(dem):.1f} à {np.nanmax(dem):.1f} m")

CRS du MNT : EPSG:26191 | EPSG reconnu : 26191
(5828, 7257) | pixel 30.0 m | valides 53% | z 0.0 à 847.6 m


## 3. Pente (méthode de Horn, en degrés)

In [3]:
def tan_pente_horn(z, cs):
    p = np.pad(z, 1, mode="edge")
    def voisin(dr, dc):
        v = p[1 + dr: p.shape[0] - 1 + dr, 1 + dc: p.shape[1] - 1 + dc]
        return np.where(np.isnan(v), z, v)          # voisin sans donnée -> valeur du pixel central
    a, b, c = voisin(-1, -1), voisin(-1, 0), voisin(-1, 1)
    d, f    = voisin(0, -1), voisin(0, 1)
    g, h, i = voisin(1, -1), voisin(1, 0), voisin(1, 1)
    dzdx = ((c + 2*f + i) - (a + 2*d + g)) / (8*cs)
    dzdy = ((g + 2*h + i) - (a + 2*b + c)) / (8*cs)
    return np.hypot(dzdx, dzdy)

tan_s = tan_pente_horn(dem, res)
tan_s[~valid] = np.nan
pente_deg = np.degrees(np.arctan(tan_s))
print(f"Pente : moyenne {np.nanmean(pente_deg):.2f}°, max {np.nanmax(pente_deg):.1f}°")

Pente : moyenne 2.89°, max 50.0°


## 4. Hydrologie : direction D8, flux (accumulation), réseau, HAND, TWI
- **Flux** = nombre de cellules drainées (accumulation D8).
- **Réseau de chenaux** = accumulation ≥ `SEUIL_RESEAU` ∪ cours d'eau OSM rastérisés.
- **HAND** = altitude de la cellule − altitude du chenal vers lequel elle s'écoule. Les cellules qui ne rejoignent aucun chenal restent vides (NaN).
- **TWI** = ln(a / tan β), avec a = aire drainée spécifique (accumulation × résolution).

In [4]:
# MNT donné à pysheds directement depuis la mémoire (pas de fichier temporaire)
z32 = np.where(valid, dem, NODATA).astype("float32")
vf = ViewFinder(affine=T, shape=dem.shape, nodata=np.float32(NODATA))
r = Raster(z32, viewfinder=vf)
grid = Grid.from_raster(r)
del z32

t0 = time.time()
pit   = grid.fill_pits(r)                 # puits résiduels
flood = grid.fill_depressions(pit)
infl  = grid.resolve_flats(flood)         # direction définie sur les zones plates
fdir  = grid.flowdir(infl)
acc   = grid.accumulation(fdir)
print(f"Hydrologie : {time.time() - t0:.0f} s")

flux = np.array(acc, dtype="float64"); flux[~valid] = np.nan

Hydrologie : 95 s


In [5]:
# Réseau de chenaux
reseau = np.nan_to_num(flux) >= SEUIL_RESEAU
if WATERWAYS.exists():
    ww = gpd.read_file(WATERWAYS)
    if ww.crs is None or ww.crs.to_epsg() != 26191:
        ww = ww.set_crs(CRS, allow_override=True) if ww.crs is None else ww.to_crs(CRS)
    ww_r = rasterize(((g, 1) for g in ww.geometry if g is not None and not g.is_empty),
                     out_shape=dem.shape, transform=T, fill=0, dtype="uint8", all_touched=True).astype(bool)
    reseau |= ww_r
    print("Waterways OSM ajoutés :", int(ww_r.sum()), "pixels")
else:
    raise FileNotFoundError("Waterways absents : le réseau de chenaux doit inclure la couche CoursEau (reproductibilité)")
reseau &= valid

# HAND
hand = np.array(grid.compute_hand(fdir, infl, Raster(reseau, viewfinder=fdir.viewfinder)), dtype="float64")
hand[~valid] = np.nan
hand[hand < 0] = 0

# TWI
a_spec = flux * res
twi = np.log(a_spec / np.maximum(tan_s, TAN_MIN))
twi[~valid] = np.nan

print(f"Chenaux : {int(reseau.sum())} pixels | HAND moy {np.nanmean(hand):.1f} m | HAND vide {np.isnan(hand[valid]).mean():.1%}")

Waterways OSM ajoutés : 68225 pixels
Chenaux : 555403 pixels | HAND moy 15.2 m | HAND vide 2.2%


## 5. Profondeur des dépressions
Le MNT corrigé est déjà comblé : la profondeur se mesure contre le MNT **brut** (comblé − brut).
Si le brut est absent, la colonne sera quasi nulle (avertissement affiché).

Les pixels dont la profondeur dépasse `DEPRESSION_MAX_M` (10 m) sont **exclus** (NaN), et non plafonnés. Ce sont des défauts du MNT brut, des carrières ou des retenues : ils ne représentent pas des cuvettes de ruissellement et, avec le min-max de l'étape 10, ils écraseraient toutes les autres mailles. Une maille dont tous les pixels sont exclus n'a pas de `depression_moy` et est retirée à l'étape 9.


In [6]:
from scipy import ndimage

def charger_brut_sans_fond():
    """Lit le MNT brut et met en NoData le fond (0 connecté au bord) et les valeurs invalides,
    comme le notebook 03, pour que la reprojection ne mélange pas le fond avec les altitudes."""
    with rasterio.open(DEM_BRUT) as s:
        raw = s.read(1).astype("float32")
        nod = s.nodata if s.nodata is not None else NODATA
        tr, crs_src = s.transform, s.crs
    inval = ~np.isfinite(raw) | (raw == nod)
    lab, _ = ndimage.label((raw == 0) & ~inval)
    bord = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])); bord = bord[bord > 0]
    fond = np.isin(lab, bord); del lab
    raw[(raw < 0) & ~inval & ~fond] = 0.0          # calage niveau marin (notebook 03)
    raw[inval | fond] = NODATA
    print(f"MNT brut : {int(fond.sum()):,} pixels de fond et {int(inval.sum()):,} invalides masqués")
    return raw, tr, crs_src

def profondeur_depressions():
    rempli = np.array(flood, dtype="float64")
    if not DEM_BRUT.exists():
        print("⚠ MNT brut absent : profondeur calculée sur le MNT corrigé (quasi nulle)")
        return np.where(valid, rempli - dem, np.nan)
    raw, tr, crs_src = charger_brut_sans_fond()
    brut = np.full(dem.shape, NODATA, dtype="float32")
    reproject(raw, brut, src_transform=tr, src_crs=crs_src, src_nodata=NODATA,
              dst_transform=T, dst_crs=prof["crs"], dst_nodata=NODATA,
              resampling=Resampling.bilinear)
    del raw
    brut = brut.astype("float64"); brut[brut == NODATA] = np.nan
    global brut_rep
    brut_rep = brut.astype("float32")          # conservé pour le tableau de contrôle
    dep = rempli - brut
    # pixels sans équivalent dans le brut (bord) : pas de profondeur calculable -> NaN, jamais une valeur inventée
    return np.where(valid & np.isfinite(dep), np.clip(dep, 0, None), np.nan)

brut_rep = None
depression_brute = profondeur_depressions()
print(f"Dépression brute : moy {np.nanmean(depression_brute):.3f} m, max {np.nanmax(depression_brute):.1f} m")

# Exclusion des pixels aberrants : au-delà du seuil, ce n'est pas une cuvette de ruissellement
# (défaut du MNT brut, carrière, retenue). Le pixel passe à NaN et la moyenne zonale se fait sur les autres pixels.
n_ecr = int((depression_brute > DEPRESSION_MAX_M).sum())
depression = np.where(depression_brute > DEPRESSION_MAX_M, np.nan, depression_brute)
print(f"Pixels exclus (> {DEPRESSION_MAX_M:.0f} m) : {n_ecr:,} ({n_ecr / max(int(valid.sum()), 1):.4%} des pixels valides)")


MNT brut : 21,625,607 pixels de fond et 0 invalides masqués
Dépression brute : moy 0.381 m, max 848.6 m
Pixels exclus (> 10 m) : 135,591 (0.6091% des pixels valides)


## 6. Export des rasters

In [7]:
def ecrire(nom, arr):
    q = prof.copy(); q.update(dtype="float32", nodata=NODATA, compress="deflate", count=1)
    with rasterio.open(OUT_R / f"{nom}.tif", "w", **q) as d:
        d.write(np.where(np.isfinite(arr), arr, NODATA).astype("float32"), 1)

RASTERS = {"pente_deg": pente_deg, "flux_acc": flux, "hand_m": hand, "twi": twi,
           "depression_m": depression, "depression_brute_m": depression_brute}
for k, v in RASTERS.items():
    ecrire(k, v)
ecrire("reseau_chenaux", reseau.astype("float64"))
print("Rasters écrits dans", OUT_R)

Rasters écrits dans C:\Users\dell\Documents\geoai-inondation-casablanca\donnees_traitees\rasters


## 7. Statistiques zonales par maille
Les mailles forment une grille régulière de 100 m : chaque pixel (30 m) est rattaché à la maille qui contient son **centre**.
C'est la convention standard des statistiques zonales (pixels dont le centre est dans la maille), sans intersection géométrique, et rapide pour plus de 2 millions de mailles.
Une maille sans pixel valide prend la valeur du pixel à son centroïde ; sinon elle reste vide (NaN).

In [8]:
import shapely

# Table Maille (étape 1) : id, taille_m, centroide (WKT), commune_id. Aucune géométrie nécessaire ici.
mailles = gpd.read_file(MAILLES, layer="maille",
                        columns=["id", "taille_m", "centroide", "commune_id"], ignore_geometry=True)

# Champ unique "centroide" (WKT) -> coordonnées x, y utilisées pour le rattachement des pixels
_pts = shapely.from_wkt(mailles["centroide"].values)
mailles["centroide_x"] = shapely.get_x(_pts)
mailles["centroide_y"] = shapely.get_y(_pts)
del _pts

# Taille lue dans la table (pas écrite en dur)
assert mailles["taille_m"].nunique() == 1, "Plusieurs tailles de maille dans la couche"
TAILLE_MAILLE = float(mailles["taille_m"].iloc[0])
assert int(TAILLE_MAILLE) == TAILLE_ATTENDUE, f"taille_m = {TAILLE_MAILLE} dans la table, {TAILLE_ATTENDUE} attendu"
print("Mailles lues :", len(mailles))

# --- Couverture du MNT : part des centroïdes dans l'emprise du raster
_g, _h = T.c, T.f
_d = T.c + T.a * dem.shape[1]
_b = T.f + T.e * dem.shape[0]
en_mnt = ((mailles["centroide_x"] >= _g) & (mailles["centroide_x"] <= _d)
          & (mailles["centroide_y"] >= _b) & (mailles["centroide_y"] <= _h))
print(f"Centroïdes dans l'emprise du MNT : {int(en_mnt.sum()):,} / {len(mailles):,} ({en_mnt.mean():.1%})")
if en_mnt.mean() < 0.99:
    print("⚠ Le MNT ne couvre pas toutes les mailles : les variables du MNT seront vides hors emprise "
          "(une maille sans valeur ne peut pas être regroupée, étape 9).")

x0 = (mailles.centroide_x - TAILLE_MAILLE/2).min()
y0 = (mailles.centroide_y - TAILLE_MAILLE/2).min()
mailles["ix"] = np.rint((mailles.centroide_x - TAILLE_MAILLE/2 - x0) / TAILLE_MAILLE).astype("int64")
mailles["iy"] = np.rint((mailles.centroide_y - TAILLE_MAILLE/2 - y0) / TAILLE_MAILLE).astype("int64")
nx, ny = int(mailles.ix.max()) + 1, int(mailles.iy.max()) + 1
assert not mailles.duplicated(["ix", "iy"]).any(), "Deux mailles sur la même case : grille non régulière"
lookup = np.full(nx * ny, -1, dtype="int64")
lookup[mailles.iy.values * nx + mailles.ix.values] = np.arange(len(mailles))

# Centre de chaque pixel -> index de maille (-1 si hors mailles)
rows, cols = np.indices(dem.shape)
xs = T.c + (cols.ravel() + 0.5) * T.a
ys = T.f + (rows.ravel() + 0.5) * T.e
ix = np.floor((xs - x0) / TAILLE_MAILLE).astype("int64")
iy = np.floor((ys - y0) / TAILLE_MAILLE).astype("int64")
dans = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
cible = np.full(ix.shape, -1, dtype="int64")
cible[dans] = lookup[iy[dans] * nx + ix[dans]]
del rows, cols, xs, ys, ix, iy, dans
print("Pixels rattachés à une maille :", int((cible >= 0).sum()))

Mailles lues : 2020705
Centroïdes dans l'emprise du MNT : 2,020,705 / 2,020,705 (100.0%)
Pixels rattachés à une maille : 22452177


In [9]:
def zonal(arr, stat):
    v = arr.ravel(); ok = (cible >= 0) & np.isfinite(v)
    g = pd.Series(v[ok]).groupby(cible[ok])
    agg = g.mean() if stat == "mean" else g.max()
    out = np.full(len(mailles), np.nan)
    out[agg.index.values] = agg.values
    return out

stats = pd.DataFrame({"id": mailles["id"].values})
stats["pente_moy"]      = zonal(pente_deg, "mean")
stats["hand_moy"]       = zonal(hand, "mean")
stats["twi_moy"]        = zonal(twi, "mean")
stats["flux_max"]       = zonal(flux, "max")
stats["depression_moy"] = zonal(depression, "mean")              # profondeur écrêtée (utilisée par le modèle)
stats["depression_moy_brute"] = zonal(depression_brute, "mean")  # non écrêtée (contrôle)
stats["depression_max_brute"] = zonal(depression_brute, "max")
stats["n_pixels"]       = np.bincount(cible[cible >= 0], minlength=len(mailles))
_ok = (cible >= 0) & valid.ravel()
stats["n_pixels_valides"] = np.bincount(cible[_ok], minlength=len(mailles))   # pixels MNT réellement valides

# Repli : valeur du pixel au centroïde pour les mailles sans pixel valide
def repli(arr, col):
    manq = stats[col].isna().values
    if not manq.any():
        return
    ci = np.floor((mailles.centroide_y.values[manq] - T.f) / T.e).astype(int)
    cj = np.floor((mailles.centroide_x.values[manq] - T.c) / T.a).astype(int)
    ok = (ci >= 0) & (ci < arr.shape[0]) & (cj >= 0) & (cj < arr.shape[1])
    vals = np.full(manq.sum(), np.nan); vals[ok] = arr[ci[ok], cj[ok]]
    stats.loc[manq, col] = vals

for col, arr in [("pente_moy", pente_deg), ("hand_moy", hand), ("twi_moy", twi),
                 ("flux_max", flux), ("depression_moy", depression),
                 ("depression_moy_brute", depression_brute), ("depression_max_brute", depression_brute)]:
    repli(arr, col)

## 8. Contrôles qualité et export des tables CSV
- Comptage des valeurs manquantes par variable (étape 9 : une maille sans valeur ne peut pas être regroupée)
- Export des champs de `Maille` (`id` + 5 variables) dans `donnees_traitees/mailles/`
- Les colonnes de contrôle sont écrites dans un fichier séparé (hors diagramme)


In [10]:
print("Mailles :", len(stats), "| valeurs manquantes par variable :")
print(stats.drop(columns=["id"]).isna().sum().to_string())
print()
print(stats.drop(columns=["id", "n_pixels", "n_pixels_valides"]).describe().T[["min", "mean", "50%", "max"]].round(3).to_string())
assert stats["id"].is_unique

# --- Diagnostics
sans_mnt = (stats.n_pixels_valides == 0)
print(f"\nMailles sans aucun pixel MNT valide (mer / hors MNT) : {int(sans_mnt.sum()):,} ({sans_mnt.mean():.1%})")
print(f"Mailles avec MNT mais HAND vide (l'eau quitte le MNT sans atteindre un chenal) : {int((~sans_mnt & stats.hand_moy.isna()).sum()):,}")
print(f"Mailles avec MNT mais depression_moy vide (tous les pixels > {DEPRESSION_MAX_M:.0f} m exclus) : {int((~sans_mnt & stats.depression_moy.isna()).sum()):,}")
q = stats.depression_moy_brute.quantile([.5, .9, .99, .999, 1]).round(2)
print("Quantiles depression_moy_brute (m) :", {k: float(v) for k, v in zip(["50%", "90%", "99%", "99.9%", "max"], q.values)})
print(f"depression_moy (pixels > {DEPRESSION_MAX_M:.0f} m exclus) : max {stats.depression_moy.max():.2f} m, moyenne {stats.depression_moy.mean():.3f} m")

# --- Export (CSV uniquement, dans le dossier des mailles)
# 1) Champs de la table Maille (diagramme) : id + 5 variables du MNT
CHAMPS_MAILLE = ["id", "pente_moy", "hand_moy", "twi_moy", "flux_max", "depression_moy"]
taille = int(TAILLE_MAILLE)
fichier_variables = OUT_T / f"variables_mnt_{taille}m.csv"
stats[CHAMPS_MAILLE].to_csv(fichier_variables, index=False)

# 2) Colonnes de contrôle (hors diagramme) : fichier séparé
fichier_controle = OUT_T / f"stats_zonales_controle_{taille}m.csv"
stats.to_csv(fichier_controle, index=False)

print("\nFichiers écrits :")
print(" -", fichier_variables)
print(" -", fichier_controle)
print("Champs de Maille :", CHAMPS_MAILLE)
stats[CHAMPS_MAILLE].head()


Mailles : 2020705 | valeurs manquantes par variable :
pente_moy               22739
hand_moy                54945
twi_moy                 22739
flux_max                22739
depression_moy          31402
depression_moy_brute    22739
depression_max_brute    22739
n_pixels                    0
n_pixels_valides            0

                        min      mean     50%          max
pente_moy             0.000     2.868   1.758       43.837
hand_moy              0.000    15.175   6.831      327.009
twi_moy               4.279     8.405   8.082       18.350
flux_max              1.000  4198.647  21.000  1758925.000
depression_moy        0.000     0.302   0.005       10.000
depression_moy_brute  0.000     0.382   0.005      846.596
depression_max_brute  0.000     0.809   0.049      847.596

Mailles sans aucun pixel MNT valide (mer / hors MNT) : 22,739 (1.1%)
Mailles avec MNT mais HAND vide (l'eau quitte le MNT sans atteindre un chenal) : 32,206
Mailles avec MNT mais depression_moy vide (to

,id,pente_moy,hand_moy,twi_moy,flux_max,depression_moy
0,2,NaN,NaN,NaN,NaN,NaN
1,6,NaN,NaN,NaN,NaN,NaN
2,7,NaN,NaN,NaN,NaN,NaN
3,8,NaN,NaN,NaN,NaN,NaN
4,10,NaN,NaN,NaN,NaN,NaN


## 9. Mailles à valeur vide : bilan pour l'étape 9 (aucun fichier écrit)

On compte les mailles utilisables (les 5 variables renseignées) et on décrit les mailles à valeur vide :
- **sans MNT** : mer ou hors du MNT, à exclure de l'analyse ;
- **HAND vide** : le chemin d'écoulement quitte le MNT (bord de la région) ou ne rejoint aucun chenal. Le HAND n'est **pas** remplacé par 0 ni par l'altitude ;
- **dépression vide** : tous les pixels dépassent `DEPRESSION_MAX_M`.

In [11]:
from scipy.spatial import cKDTree

VARIABLES = CHAMPS_MAILLE[1:]                         # les 5 variables du MNT
complet = stats[VARIABLES].notna().all(axis=1).values
sans_mnt = (stats["n_pixels_valides"] == 0).values
hand_vide = stats["hand_moy"].isna().values & ~sans_mnt
dep_vide = stats["depression_moy"].isna().values & ~sans_mnt

print(f"Mailles utilisables (5 variables renseignées) : {complet.sum():,} / {len(stats):,} ({complet.mean():.1%})")
print(f"  - sans MNT (mer / hors MNT)           : {sans_mnt.sum():,}")
print(f"  - MNT valide mais HAND vide           : {hand_vide.sum():,}")
print(f"  - MNT valide mais dépression vide     : {dep_vide.sum():,}")

# Communes les plus concernées par le HAND vide
print("\nHAND vide, par commune :")
print(mailles.loc[hand_vide, "commune_id"].value_counts().head(10).to_string())

# Les mailles à HAND vide sont-elles au bord du MNT (côte, limite de la région) ?
if sans_mnt.any() and hand_vide.any():
    xy = mailles[["centroide_x", "centroide_y"]].values
    arbre = cKDTree(xy[sans_mnt])
    dist_bord, _ = arbre.query(xy[hand_vide])
    alt_moy = zonal(dem, "mean")
    print(f"\nHAND vide à moins de 1 km d'une maille sans MNT : {(dist_bord <= 1000).mean():.1%}")
    print("Altitude moyenne des mailles à HAND vide (m) :")
    print(pd.Series(alt_moy[hand_vide]).describe(percentiles=[.5, .9]).round(1).to_string())

Mailles utilisables (5 variables renseignées) : 1,957,212 / 2,020,705 (96.9%)
  - sans MNT (mer / hors MNT)           : 22,739
  - MNT valide mais HAND vide           : 32,206
  - MNT valide mais dépression vide     : 8,663

HAND vide, par commune :
commune_id
29     1890
65     1732
23     1355
78     1331
35     1153
62      947
63      869
113     855
69      841
90      774

HAND vide à moins de 1 km d'une maille sans MNT : 54.2%
Altitude moyenne des mailles à HAND vide (m) :
count    32206.0
mean       195.7
std        189.8
min          0.0
50%        155.8
90%        425.4
max        842.8
